In [ ]:

from fastapi import (
    APIRouter,
    HTTPException,
    status,
    Depends,
)

from app.api.auth import get_current_user


router = APIRouter(
    prefix="/growth",
    tags=["Growth"],
)


def get_database():
    from app.main import app

    database = getattr(app.state, "database", None)

    if database is None:
        raise RuntimeError("Database is not initialized.")

    return database


def verify_project_ownership(
    database,
    project_id: str,
    user_id: str,
):
    project = database.collection("projects").find_one(
        {
            "id": project_id,
            "user_id": user_id,
        }
    )

    if project is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Project not found.",
        )

    return project


def clean(document):
    result = dict(document)
    result.pop("_id", None)
    return result


@router.get("/project/{project_id}")
async def get_project_growth(
    project_id: str,
    current_user=Depends(get_current_user),
):
    """
    Return a learner growth summary based on persisted
    mastery, assessments, activities and recommendations.
    """

    database = get_database()

    verify_project_ownership(
        database,
        project_id,
        current_user.id,
    )

    mastery = list(
        database.collection("mastery").find(
            {
                "project_id": project_id,
                "user_id": current_user.id,
            }
        )
    )

    assessments = list(
        database.collection("assessments").find(
            {
                "project_id": project_id,
                "user_id": current_user.id,
            }
        )
    )

    activities_count = database.collection(
        "activities"
    ).count_documents(
        {
            "project_id": project_id,
            "user_id": current_user.id,
        }
    )

    recommendations_count = database.collection(
        "recommendations"
    ).count_documents(
        {
            "project_id": project_id,
            "user_id": current_user.id,
        }
    )

    completed_assessments = [
        assessment
        for assessment in assessments
        if assessment.get("status") == "completed"
    ]

    average_mastery = 0.0

    if mastery:
        average_mastery = sum(
            float(item.get("score", 0.0))
            for item in mastery
        ) / len(mastery)

    average_assessment_score = 0.0

    scored_assessments = [
        assessment
        for assessment in completed_assessments
        if assessment.get("score") is not None
        and assessment.get("max_score")
    ]

    if scored_assessments:
        values = []

        for assessment in scored_assessments:
            max_score = float(
                assessment.get("max_score", 0)
            )

            if max_score > 0:
                values.append(
                    float(assessment.get("score", 0))
                    / max_score
                )

        if values:
            average_assessment_score = sum(values) / len(values)

    trend_counts = {
        "improving": 0,
        "stable": 0,
        "needs_attention": 0,
    }

    for item in mastery:
        trend = item.get("trend", "stable")

        if trend in trend_counts:
            trend_counts[trend] += 1

    strong_concepts = [
        item.get("concept_id")
        for item in mastery
        if float(item.get("score", 0.0)) >= 0.80
    ]

    weak_concepts = [
        item.get("concept_id")
        for item in mastery
        if float(item.get("score", 0.0)) < 0.50
    ]

    concept_items = [
        {
            "concept_id": item.get("concept_id"),
            "score": round(float(item.get("score", 0.0)), 4),
            "trend": item.get("trend", "stable"),
            "confidence": round(float(item.get("confidence", 0.0)), 4),
        }
        for item in sorted(
            mastery,
            key=lambda value: float(value.get("score", 0.0)),
        )
    ]

    return {
        "project_id": project_id,
        "mastery": {
            "concept_count": len(mastery),
            "average_score": round(
                average_mastery,
                4,
            ),
            "trend_counts": trend_counts,
            "strong_concepts": strong_concepts,
            "weak_concepts": weak_concepts,
            "concepts": concept_items,
        },
        "assessments": {
            "total": len(assessments),
            "completed": len(completed_assessments),
            "average_score": round(
                average_assessment_score,
                4,
            ),
        },
        "activity": {
            "event_count": activities_count,
        },
        "recommendations": {
            "count": recommendations_count,
        },
    }
